# Data Validation (Pandera & Great Expectations)

In [ ]:
# Day 36 — Data Validation
# Garbage In, Garbage Out! In production MLOps, most model failures
# are NOT bugs in the ML algorithm — they are bad/corrupted input data:
#   - Negative CPU values (-15%)
#   - Nulls sneaking into required columns
#   - Wrong column types (strings instead of floats)
#   - Unexpected categories
#
# Pydantic validates single JSON payloads. For entire DataFrames/datasets,
# we use Pandera (lightweight, Pythonic) or Great Expectations (enterprise suites).
#
# Today we'll cover:
#   1. Pandera DataFrameSchema & Column checks
#   2. Custom validation rules & lazy validation
#   3. @pa.check_types decorator for pipeline functions
#   4. Great Expectations concepts (Expectation Suites)
#   5. Failing CI/CD / training pipelines on bad data
#
# Install: pip install pandera great-expectations

### 1. Pandera — DataFrameSchema & Column Checks

In [ ]:
import pandas as pd
import pandera as pa
from pandera import Column, DataFrameSchema, Check

# Define the expected contract for our server metrics dataset
server_schema = DataFrameSchema({
    "server_id": Column(str, nullable=False, unique=True),
    "cpu": Column(float, checks=[Check.ge(0.0), Check.le(100.0)], nullable=False),
    "memory": Column(float, checks=[Check.in_range(0.0, 100.0)], nullable=False),
    "env": Column(str, checks=Check.isin(["prod", "staging", "dev"])),
    "error_count": Column(int, checks=Check.ge(0), coerce=True)
})

# Valid DataFrame
good_df = pd.DataFrame({
    "server_id": ["web-01", "web-02", "db-01"],
    "cpu": [45.5, 88.0, 62.3],
    "memory": [60.0, 82.1, 75.0],
    "env": ["prod", "prod", "staging"],
    "error_count": [0, 12, 2]
})

validated_df = server_schema.validate(good_df)
print("Good data passed validation!")
print(validated_df)

### 2. Catching Bad Data & Lazy Validation

In [ ]:
# What happens when bad data arrives?
# lazy=True collects ALL errors at once instead of stopping at the first one.

bad_df = pd.DataFrame({
    "server_id": ["web-01", "web-01", "db-01"],   # duplicate server_id!
    "cpu": [45.5, 150.0, -5.0],                    # >100 and <0!
    "memory": [60.0, 82.1, 75.0],
    "env": ["prod", "qa", "staging"],              # 'qa' not allowed!
    "error_count": [0, -1, 2]                      # negative errors!
})

try:
    server_schema.validate(bad_df, lazy=True)
except pa.errors.SchemaErrors as err:
    print("Data quality gate BLOCKED bad data!")
    print(err.failure_cases[["column", "check", "failure_case"]])

### 3. Decorator-Based Validation & Great Expectations Overview

In [ ]:
# You can also decorate pipeline functions with @pa.check_input / @pa.check_output

@pa.check_input(server_schema)
def compute_avg_cpu(df: pd.DataFrame) -> float:
    return float(df["cpu"].mean())

print("Average CPU:", compute_avg_cpu(good_df))

# --- Great Expectations (GE) ---
# When you need rich HTML data docs, S3/SQL data connectors, and team-wide
# Expectation Suites, Great Expectations is the enterprise standard:
#
#   import great_expectations as gx
#   context = gx.get_context()
#   validator = context.sources.pandas_default.read_dataframe(good_df)
#   validator.expect_column_values_to_be_between("cpu", min_value=0, max_value=100)
#   validator.expect_column_values_to_not_be_null("server_id")
#   results = validator.validate()
#
# Rule of thumb:
#   Pandera            -> Fast, code-first, great inside Python/Pytest/Prefect
#   Great Expectations -> Great for data engineering teams & HTML audit reports

### 4. Validating Real Dirty Data (`datasets/cloud_costs.csv`)

In [ ]:
# Let us test Pandera against our real dirty dataset `datasets/cloud_costs.csv`!
# It contains a duplicate row, a NaN cost_usd, and a negative cost (-12.00).

import pandas as pd
import pandera as pa
from pandera import Column, DataFrameSchema, Check

cost_schema = DataFrameSchema({
    "date": Column(str, nullable=False),
    "team": Column(str, nullable=False),
    "cost_usd": Column(float, checks=Check.ge(0.0), nullable=False),
    "usage_hours": Column(float, checks=Check.ge(0.0), nullable=False)
}, unique=["date", "team", "env", "service"])

raw_costs = pd.read_csv("datasets/cloud_costs.csv")
try:
    cost_schema.validate(raw_costs, lazy=True)
except pa.errors.SchemaErrors as err:
    print("Caught real data quality issues in cloud_costs.csv:")
    print(err.failure_cases[["column", "check", "failure_case"]])

In [ ]:
# Practice Questions:
#
# 1. Why is `lazy=True` better than default validation in an ETL/ML pipeline?
# 2. What is the difference between Pydantic (Day 42) and Pandera (Day 36)?
# 3. Why should data validation run BEFORE model training in CI/CD?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: Kubernetes Pod Metrics Validator
# Build a Pandera DataFrameSchema for pod metrics with columns:
# - pod_name (str, not null, matches regex '^[a-z0-9-]+$')
# - cpu_millicores (int, >= 0, <= 64000)
# - memory_mb (float, > 0)
# - restarts (int, >= 0)

# Exercise 2: ML Training Data Quality Gate
# Write a function `validate_training_data(df)` that checks:
# - No duplicate rows
# - Null percentage in any column < 5%
# - Target column has at least 10% positive class (no extreme imbalance)

# Exercise 3: CI/CD Validation Step
# Write a CLI-ready script `validate_gate.py` that reads a CSV,
# runs Pandera validation, and exits with code 1 (sys.exit(1)) if validation fails.